In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

P = "../models/transformer/distilbert"
tok = AutoTokenizer.from_pretrained(P)
model = AutoModelForSequenceClassification.from_pretrained(P).eval()

def predict(text):
    enc = tok(text, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        prob_fake = torch.softmax(model(**enc).logits, dim=1)[0, 1].item()
    return ("Fake" if prob_fake >= 0.5 else "Real"), round(max(prob_fake, 1 - prob_fake), 4)

# Test 1: 10 validation articles with known answers
df = pd.read_csv("../data/interim/master.csv", usecols=["row_id", "label", "raw_text"])
df = df.merge(pd.read_csv("../data/splits/split.csv"), on="row_id")
sample = df[df["split"] == "val"].sample(10, random_state=42)
del df
print("--- validation sample ---")
for _, r in sample.iterrows():
    true = "Fake" if r["label"] == 1 else "Real"
    print(f"true={true} | pred={predict(r['raw_text'])} | {r['raw_text'][:50]}")

# Test 2: text where style and truth disagree (hand-written)
probes = {
    "false claim, wire style": "WASHINGTON (Reuters) - The U.S. government confirmed on Tuesday that the moon landing was staged, officials said.",
    "true-sounding, blog style": "Watch this video: the Senate passed the budget bill today. Featured image via Twitter. Read more!",
    "neutral, no source cues": "The city council will meet next week to discuss the new parking rules.",
}
print("--- probes ---")
for name, t in probes.items():
    print(name, "->", predict(t))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--- validation sample ---
true=Real | pred=('Real', 1.0) | Bangladesh police arrest militant suspected of kil
true=Fake | pred=('Fake', 0.9999) | Bloomberg Has Some News About Running For Presiden
true=Fake | pred=('Fake', 0.9999) | KARMA: Federal Judge Slaps Texas GOP Bush Donor Wi
true=Fake | pred=('Fake', 0.9999) | DEMOCRAT SENATOR Makes Up FAKE Anti-Trump Story Du
true=Fake | pred=('Fake', 0.9999) | NRA Had This To Say After ANOTHER School Shooting 
true=Fake | pred=('Fake', 0.9998) | WHY DID TEXAS SHOOTER Shave His Beard Only Days Be
true=Fake | pred=('Fake', 0.9999) | Anti-Gay Pastor Who Cheered Killing Of Pulse Victi
true=Real | pred=('Real', 1.0) | Trump transgender decision 'shocking and ignorant'
true=Fake | pred=('Fake', 0.9999) | Trump Gets CHEWED OUT After Insulting Obamacare Pa
true=Real | pred=('Real', 1.0) | North Korea says Trump begged for a war during his
--- probes ---
false claim, wire style -> ('Real', 1.0)
true-sounding, blog style -> ('Fake', 0.9996)
neutral, no

In [2]:
import sys
sys.path.append("..")
from src.baseline.predict_baseline import BaselinePredictor

base = BaselinePredictor(version="v0_raw")
for name, t in probes.items():
    r = base.predict(t)
    print(f"{name} | baseline: {r['label']} ({r['confidence']:.3f}) | distilbert: {predict(t)}")

false claim, wire style | baseline: Real (0.998) | distilbert: ('Real', 1.0)
true-sounding, blog style | baseline: Fake (0.999) | distilbert: ('Fake', 0.9996)
neutral, no source cues | baseline: Real (0.751) | distilbert: ('Fake', 0.9876)
